# Workflow Capacity Explorer

Сравнение **монолит vs sharding**: wait / work / total по срезам.

- Jobs кэшируются в `data/cache/` — смена квот не требует перевыкачки
- Конфиг: `config/capacity.example.yml` (`static_runners`, `footprints`, quotas)
- Sankey: перераспределение wait/work между конфигами и режимами

In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from workflow_capacity.cache import ensure_dataset, list_datasets, load_dataset
from workflow_capacity.config import PoolConfig
from workflow_capacity.compare import (
    evaluate_config,
    results_to_dataframe,
    sankey_wait_work_flow,
    sankey_compare_configs,
)
from workflow_capacity.metrics import D_GROUPS, comparison_table, aggregate_p90, scenario_metrics
from workflow_capacity.pr_check import build_pr_check_runs
from workflow_capacity.simulate import run_pair

CONFIG_PATH = ROOT / 'config' / 'capacity.example.yml'
CACHE_DIR = ROOT / 'data' / 'cache'
PEAK_HOURS = list(range(9, 16))

## 1. Исторические данные (кэш)

Первый запуск с `refresh=True` скачает jobs с GitHub (`gh auth login`).
Дальше файлы лежат рядом: `jobs_ydb-platform_ydb_YYYY-MM-DD_YYYY-MM-DD.json`.

In [ ]:
DAYS = 14
REPO = 'ydb-platform/ydb'
REFRESH = False  # True — перекачать с GitHub

cached = list_datasets(CACHE_DIR)
print('Cached windows:')
for ds in cached:
    print(f'  - {ds.path.name} ({len(ds.jobs)} jobs, {ds.since[:10]} .. {ds.until[:10]})')

# Выберите окно: None = ensure_dataset(days=DAYS), иначе путь к файлу из списка выше
SELECTED_CACHE = None  # например: CACHE_DIR / 'jobs_ydb-platform_ydb_2026-05-30_2026-06-13.json'

if SELECTED_CACHE is not None:
    dataset = load_dataset(SELECTED_CACHE)
else:
    dataset = ensure_dataset(days=DAYS, repo=REPO, cache_dir=CACHE_DIR, refresh=REFRESH)

print(f'\nActive: {dataset.path.name}')
print(f'Jobs: {len(dataset.jobs)} | {dataset.since[:10]} .. {dataset.until[:10]}')

### Сравнение нескольких окон истории

Разные окна (7d / 14d / 30d) лежат в `data/cache/` рядом. Пересчёт квот и конфигов **не требует** перевыкачки — только выберите другой файл.

In [ ]:
# Сравнить два кэшированных окна на одном конфиге (без повторного сбора)
COMPARE_WINDOWS = False
WINDOW_A = None  # путь к jobs_*.json или None = текущий dataset
WINDOW_B = None  # второй файл, например 7d окно

if COMPARE_WINDOWS and WINDOW_A and WINDOW_B:
    ds_a = load_dataset(WINDOW_A)
    ds_b = load_dataset(WINDOW_B)
    cfg_cmp = PoolConfig.load(CONFIG_PATH, name='current')
    pr_a = build_pr_check_runs(ds_a.jobs, classify=False)
    pr_b = build_pr_check_runs(ds_b.jobs, classify=False)
    ra = evaluate_config(ds_a.jobs, pr_a, cfg_cmp, peak_hours=PEAK_HOURS)
    rb = evaluate_config(ds_b.jobs, pr_b, cfg_cmp, peak_hours=PEAK_HOURS)
    cmp_df = pd.DataFrame([
        {
            'window': ds_a.path.stem,
            'jobs': len(ds_a.jobs),
            'mono_total_p90': ra.base_agg[(None, 'all')]['total_p90'],
            'shard_total_p90': ra.par_agg[(None, 'all')]['total_p90'],
            'delta_total': ra.par_agg[(None, 'all')]['total_p90'] - ra.base_agg[(None, 'all')]['total_p90'],
        },
        {
            'window': ds_b.path.stem,
            'jobs': len(ds_b.jobs),
            'mono_total_p90': rb.base_agg[(None, 'all')]['total_p90'],
            'shard_total_p90': rb.par_agg[(None, 'all')]['total_p90'],
            'delta_total': rb.par_agg[(None, 'all')]['total_p90'] - rb.base_agg[(None, 'all')]['total_p90'],
        },
    ]).round(1)
    display(cmp_df)
else:
    print('Set COMPARE_WINDOWS=True and WINDOW_A/WINDOW_B paths to compare cached windows.')

## 2. Конфигурации capacity

Базовый конфиг + сценарии масштабирования квот.
Редактируйте `config/capacity.example.yml` для static runners и footprints.

In [ ]:
base = PoolConfig.load(CONFIG_PATH, name='current')

CONFIGS = [
    base,
    base.with_quota_scale(vcpu=1.1, name='vcpu+10%'),
    base.with_quota_scale(vcpu=1.25, instances=1.25, ram_gb=1.25, nrd_ssd_gb=1.25, name='all+25%'),
    base.with_quota_scale(vcpu=2.0, instances=2.0, ram_gb=2.0, nrd_ssd_gb=2.0, name='all x2'),
]

pd.DataFrame([
    {
        'config': c.name,
        'instances': int(c.quotas['instances']),
        'vcpu': int(c.quotas['vcpu']),
        'ram_gb': int(c.quotas['ram_gb']),
        'vm_budget': round(c.max_instances_budget(), 1),
    }
    for c in CONFIGS
])

## 3. Прогон симуляций

`classify=False` — быстрее (без gh API на каждый PR). Для точного classifier поставьте `True`.

In [ ]:
CLASSIFY = False
ROLLOUT = 'all eligible'  # или main + stable/*, main only

from workflow_capacity.metrics import ROLL_OUTS
rollout = next(r for r in ROLL_OUTS if r[1] == ROLLOUT)
_, rollout_label, shard_eligible = rollout

pr_runs = build_pr_check_runs(dataset.jobs, classify=CLASSIFY)
results = []
for cfg in CONFIGS:
    results.append(
        evaluate_config(
            dataset.jobs,
            pr_runs,
            cfg,
            rollout_label=rollout_label,
            shard_eligible=shard_eligible,
            peak_hours=PEAK_HOURS,
        )
    )

df = results_to_dataframe(results)
summary = df.groupby('config').agg(
    mono_wait=('mono_wait_p90', 'median'),
    mono_work=('mono_work_p90', 'median'),
    mono_total=('mono_total_p90', 'median'),
    shard_wait=('shard_wait_p90', 'median'),
    shard_work=('shard_work_p90', 'median'),
    shard_total=('shard_total_p90', 'median'),
    delta_total=('delta_total', 'median'),
).round(1)
display(summary)

## 4. Таблица по срезам (час × D)

In [ ]:
CONFIG_NAME = 'current'
D_FILTER = 'все D'

view = df[(df['config'] == CONFIG_NAME) & (df['d_group'] == D_FILTER)].copy()
cols = [
    'hour_utc', 'd_group', 'n',
    'mono_wait_p90', 'shard_wait_p90', 'delta_wait',
    'mono_work_p90', 'shard_work_p90', 'delta_work',
    'mono_total_p90', 'shard_total_p90', 'delta_total', 'delta_total_pct',
]
display(view[cols].round(1))

## 5. Sankey: монолит → шардинг (wait / work)

In [ ]:
item = next(r for r in results if r.config.name == CONFIG_NAME)
row = item.base_agg.get((None, 'all'), {})
prow = item.par_agg.get((None, 'all'), {})

fig = go.Figure(data=[go.Sankey(
    node=dict(
        label=['Монолит wait', 'Монолит work', 'Шардинг wait', 'Шардинг work'],
        color=['#e45756', '#4c78a8', '#f58518', '#72b7b2'],
    ),
    link=dict(
        source=[0, 1],
        target=[2, 3],
        value=[max(prow.get('wait_p90') or 0, 0.1), max(prow.get('work_p90') or 0, 0.1)],
    ),
)])
fig.update_layout(
    title=f'{CONFIG_NAME}: p90 wait/work (все D, пик 09-15) — '
            f"mono total {row.get('total_p90', 0):.0f} → shard {prow.get('total_p90', 0):.0f} min",
    height=420,
)
fig.show()

## 6. Sankey: сравнение двух конфигов (sharding total)

In [ ]:
LEFT = 'current'
RIGHT = 'all+25%'
HOUR = 10
D_KEY = 'all'

left = next(r for r in results if r.config.name == LEFT)
right = next(r for r in results if r.config.name == RIGHT)
sk = sankey_compare_configs(left, right, hour=HOUR, d_key=D_KEY)

fig = go.Figure(data=[go.Sankey(node=dict(label=sk['node']['label']), link=sk['link'])])
fig.update_layout(
    title=f'Sharding p90 @ {HOUR:02d}:00, D={D_KEY}: {LEFT} total {sk["meta"]["left_total"]:.0f} min',
    height=380,
)
fig.show()
print(f'{RIGHT}: wait {sk["meta"]["right_wait"]:.1f}, work {sk["meta"]["right_work"]:.1f}, total {sk["meta"]["right_total"]:.1f} min')

## 7. Интерактивные слайдеры квот

In [ ]:
import ipywidgets as w

vcpu_mult = w.FloatSlider(value=1.0, min=0.5, max=2.5, step=0.05, description='vCPU x')
inst_mult = w.FloatSlider(value=1.0, min=0.5, max=2.5, step=0.05, description='VM max x')
ram_mult = w.FloatSlider(value=1.0, min=0.5, max=2.5, step=0.05, description='RAM x')
ssd_mult = w.FloatSlider(value=1.0, min=0.5, max=2.5, step=0.05, description='SSD x')
out = w.Output()


def rerun(_=None):
    cfg = base.with_quota_scale(
        vcpu=vcpu_mult.value,
        instances=inst_mult.value,
        ram_gb=ram_mult.value,
        nrd_ssd_gb=ssd_mult.value,
        name='interactive',
    )
    item = evaluate_config(
        dataset.jobs, pr_runs, cfg,
        rollout_label=rollout_label,
        shard_eligible=shard_eligible,
        peak_hours=PEAK_HOURS,
    )
    rows = []
    for d_key, d_label in D_GROUPS:
        b = item.base_agg.get((None, d_key), {})
        p = item.par_agg.get((None, d_key), {})
        rows.append({
            'd_group': d_label,
            'mono_wait': b.get('wait_p90'),
            'shard_wait': p.get('wait_p90'),
            'mono_total': b.get('total_p90'),
            'shard_total': p.get('total_p90'),
            'delta_total': (p.get('total_p90') or 0) - (b.get('total_p90') or 0),
        })
    with out:
        out.clear_output(wait=True)
        display(pd.DataFrame(rows).round(1))
        print(
            f"vm_budget={cfg.max_instances_budget():.1f}, "
            f"vcpu={int(cfg.quotas['vcpu'])}, "
            f"ram={int(cfg.quotas['ram_gb'])}, ssd={int(cfg.quotas['nrd_ssd_gb'])}"
        )

for slider in (vcpu_mult, inst_mult, ram_mult, ssd_mult):
    slider.observe(rerun, names='value')

display(w.VBox([vcpu_mult, inst_mult, ram_mult, ssd_mult, out]))
rerun()